# Custom PDF integrators

Binned cost functions like `BinnedNLL` and `ExtendedBinnedNLL` normally require a CDF (or a scaled CDF for the extended case). When only the PDF is available, iminuit can compute the per-bin integrals for you via the `use_pdf` keyword:

| `use_pdf` | Method | Accuracy | Speed | Dimensions |
|-----------|--------|----------|-------|------------|
| `"approximate"` | PDF at bin centre × bin width | Low for coarse bins | Fast | Any |
| `"numerical"` | `scipy.integrate.quad` per bin | High | Slow | 1D only |
| `"custom"` | User-supplied integrator | Up to you | Up to you | Any |

The `"custom"` option lets you choose any quadrature rule, e.g. [Simpson's rule](https://en.wikipedia.org/wiki/Simpson%27s_rule). This example sits comfortably between the two built-in modes in terms of both accuracy and speed, and can also works for multi-dimensional histograms.

## Integrator interface

The integrator is a callable with the signature

```python
def integrator(pdf, a, b) -> float:
    ...
```

where

* `pdf` is the model PDF with the fit parameters already bound — you only need to pass sample points to it.
* `a`, `b` are the bin edges.  
  * For 1D histograms they are scalars.  
  * For N-D histograms they are 1-D arrays of length N (the lower and upper corners of the bin hyper-rectangle).

The integrator must return the integral of `pdf` over `[a, b]` as a scalar float.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.integrate import simpson
from scipy.stats import norm

from iminuit import Minuit
from iminuit.cost import BinnedNLL, ExtendedBinnedNLL

%config InlineBackend.figure_formats = ['svg']

## 1D example

We generate 1000 samples from a Gaussian and bin them. The histogram range is wide enough to capture the full distribution, but the bins are coarse enough (bin width = σ) to make the rectangle approximation error visible.

In [ ]:
rng = np.random.default_rng(1)
data = rng.normal(0.5, 0.2, size=1000)

# Range captures ±5σ so tails are not clipped; bin width = 0.2 = σ
n, xe = np.histogram(data, bins=10, range=(-0.5, 1.5))
cx = 0.5 * (xe[1:] + xe[:-1])

plt.stairs(n, xe)
plt.xlabel("x")
plt.ylabel("counts");

We define a model PDF. Assume that we only have the PDF and no closed-form CDF.

In [ ]:
def model_pdf(x, mu, sigma):
    return norm.pdf(x, mu, sigma)

### Comparing the three modes

We compare the raw per-bin integrals produced by each method against the analytical CDF reference. This is independent of the cost function and directly shows the integration accuracy.

In [ ]:
def model_cdf(x, mu, sigma):
    return norm.cdf(x, mu, sigma)


truth = (0.5, 0.2)


# Custom: Simpson's rule with 11 evaluation points per bin
def simpson_integrator(pdf, a, b):
    x = np.linspace(a, b, 11)
    return simpson(pdf(x), x=x)


# Cost functions used for timing and fitting below
c_approx = BinnedNLL(n, xe, model_pdf, use_pdf="approximate")
c_numerical = BinnedNLL(n, xe, model_pdf, use_pdf="numerical")
c_custom = BinnedNLL(n, xe, model_pdf, use_pdf="custom", integrator=simpson_integrator)

We now compute the per-bin integrals of each method directly and compare them to the analytical CDF differences.

In [ ]:
from scipy.integrate import quad as scipy_quad

dx = np.diff(xe)

# Analytical reference
ref = np.diff(model_cdf(xe, *truth))

# Approximate: pdf at bin midpoint times bin width
approx_int = model_pdf(cx, *truth) * dx

# Numerical: scipy.integrate.quad
quad_int = np.array(
    [
        scipy_quad(lambda x: model_pdf(x, *truth), xe[i], xe[i + 1])[0]
        for i in range(len(xe) - 1)
    ]
)

# Custom: Simpson's rule with 11 points
custom_int = np.array(
    [
        simpson(
            model_pdf(np.linspace(xe[i], xe[i + 1], 11), *truth),
            x=np.linspace(xe[i], xe[i + 1], 11),
        )
        for i in range(len(xe) - 1)
    ]
)

fig, ax = plt.subplots()
ax.plot(cx, approx_int - ref, "o-", label="approximate")
ax.plot(cx, quad_int - ref, "s-", label="numerical (quad)")
ax.plot(cx, custom_int - ref, "^-", label="custom (Simpson)")
ax.axhline(0, color="k", lw=0.8, ls="--")
ax.set_xlabel("bin centre")
ax.set_ylabel("integral -- reference")
ax.legend()
ax.set_title("Per-bin integration error relative to analytical CDF");

The approximate method has a visible systematic error (positive near the peak where the PDF is concave, negative in the wings). Both `quad` and Simpson's rule match the reference to near machine precision, while Simpson is significantly faster.

### Timing comparison

In [ ]:
print("approximate:")
%timeit c_approx.prediction(truth)
print("numerical (quad):")
%timeit c_numerical.prediction(truth)
print("custom (Simpson, 11 points):")
%timeit c_custom.prediction(truth)

### Fitting with the custom integrator

The cost function is used with `Minuit` exactly as any other `BinnedNLL`.

In [ ]:
m = Minuit(c_custom, mu=0.5, sigma=0.2)
m.limits["sigma"] = (0, None)
m.migrad()
m.hesse()
m

## ExtendedBinnedNLL

`ExtendedBinnedNLL` works in the same way. The model must now be a *scaled* PDF —- one that returns expected counts rather than a normalised density.

In [ ]:
def scaled_pdf(x, n_total, mu, sigma):
    return n_total * norm.pdf(x, mu, sigma)


c_ext = ExtendedBinnedNLL(
    n, xe, scaled_pdf, use_pdf="custom", integrator=simpson_integrator
)

m_ext = Minuit(c_ext, n_total=n.sum(), mu=0.5, sigma=0.2)
m_ext.limits["n_total", "sigma"] = (0, None)
m_ext.migrad()
m_ext.hesse()
m_ext

## 2D example

The `"custom"` mode is the only `use_pdf` option that supports multi-dimensional histograms with user-controlled accuracy. For N-D histograms, `a` and `b` passed to the integrator are 1-D arrays of length N representing the lower and upper corners of the bin hyper-rectangle, and `pdf(x)` expects `x` with shape `(N, n_points)`.

In [ ]:
# Generate 2D data: independent Gaussians along each axis
data2d = rng.normal([0.5, 0.3], [0.15, 0.1], size=(2000, 2))
n2, xe2, ye2 = np.histogram2d(
    data2d[:, 0], data2d[:, 1], bins=(8, 8), range=((0, 1), (0, 0.6))
)


def pdf_2d(xy, mux, muy, sx, sy):
    x, y = xy
    return norm.pdf(x, mux, sx) * norm.pdf(y, muy, sy)


def simpson_integrator_2d(pdf, a, b):
    """Double Simpson's rule for a rectangular 2D bin."""
    x0 = np.linspace(a[0], b[0], 11)
    x1 = np.linspace(a[1], b[1], 11)
    X0, X1 = np.meshgrid(x0, x1, indexing="ij")
    pts = np.array([X0.ravel(), X1.ravel()])
    y = pdf(pts).reshape(11, 11)
    return simpson(simpson(y, x=x1, axis=1), x=x0)


c2d = BinnedNLL(
    n2, (xe2, ye2), pdf_2d, use_pdf="custom", integrator=simpson_integrator_2d
)

m2d = Minuit(c2d, mux=0.5, muy=0.3, sx=0.15, sy=0.1)
m2d.limits["sx", "sy"] = (0, None)
m2d.migrad()
m2d.hesse()
m2d

The fitted values are close to the true values `(0.5, 0.3, 0.15, 0.1)` and within $2\sigma$ error.

## Writing your own integrator

Any quadrature rule can be used. The only requirement is the signature `integrator(pdf, a, b) -> float`. Some examples:

```python
# Gauss-Legendre quadrature (high accuracy, fixed number of evaluations)
from numpy.polynomial.legendre import leggauss

nodes, weights = leggauss(10)  # pre-compute once outside the integrator

def gauss_legendre(pdf, a, b):
    # transform nodes from [-1, 1] to [a, b]
    mid = 0.5 * (b + a)
    half = 0.5 * (b - a)
    x = mid + half * nodes
    return half * np.dot(weights, pdf(x))


# Adaptive quadrature from scipy (most accurate, slowest)
from scipy.integrate import quad

def adaptive_quad(pdf, a, b):
    return quad(pdf, a, b)[0]
```

For N-D histograms, `a` and `b` are arrays, so scale the quadrature rule per dimension, as shown in `simpson_integrator_2d` above.

A useful rule of thumb: start with a small number of Simpson points (e.g. 5 or 7) and increase until the fit result stops changing. For smooth PDFs, 11 points is usually more than sufficient for typical binning.